In [1]:
# Install dependencies (run once)
%pip install shap missforest openpyxl joblib --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,PowerTransformer,QuantileTransformer,MinMaxScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import r2_score, root_mean_squared_error
from missforest import MissForest
from sklearn.ensemble import RandomForestRegressor
import shap as s
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import cross_val_predict
from scipy.optimize import differential_evolution
from scipy.spatial.distance import cdist
from itertools import combinations
import random
from joblib import Parallel, delayed
import multiprocessing
import optuna as op
# Reproducibility
RANDOM_STATE = 62

# Column config — centralised so changes propagate everywhere
TARGET_COL   = 'f_SAR_W_g'
CAT_COLS      = ['c_shape_type', 'c_magnetic_phase', 'c_coating']
DROP_COLS     = ['-', 'DOI', 'c_core_composition', 'c_dopant_element', 'f_dopant_ratio']
FEATURE_NAMES = [
    'f_avg_core_width_nm', 'f_Ms_emu_g_MNP', 'f_Hc_Oe', 'is_doped',
    'c_coating', 'f_H_Oe', 'f_FCY_kHz', 'f_conc_MNP_mg_mL',
    'c_shape_type', 'c_magnetic_phase', 'f_avg_core_sd_nm _BOTH',
    'Ca_comp', 'Co_comp', 'Cu_comp', 'Fe_comp', 'Ga_comp', 'Gd_comp',
    'Mg_comp', 'Mn_comp', 'Nd_comp', 'Ni_comp', 'Sn_comp', 'Tb_comp',
    'Zn_comp', 'f_magnetic_anisotropy', 'f_magnetic_dipole_moment',
]

In [3]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        c.replace('\ufeff', '').replace('\u200b', '').strip()
        for c in df.columns
    ]
    return df


df_raw = pd.read_excel("/kaggle/input/datasets/magneticmaterials/hyperthermia/Final.xlsx")
df_raw = clean_column_names(df_raw)

# Drop identifier/non-feature columns that exist in this dataset
cols_to_drop = [c for c in DROP_COLS if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop)

print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded 1548 rows × 33 columns


,S. No.,f_avg_core_width_nm,f_Ms_emu_g_MNP,f_Hc_Oe,is_doped,f_is_coated,f_is_organic,f_is_inorganic,f_is_composite_coating,c_coating,...,Fe_comp,Ga_comp,Gd_comp,Mg_comp,Mn_comp,Nd_comp,Ni_comp,Sn_comp,Tb_comp,Zn_comp
0,1,15.0,60.00,0.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
1,2,17.0,36.78,6.0,1,1,1,0,0,Chitosan,...,2.93,0.0,0.07,0.0,0.0,0.0,0.0,0,0.0,0.0
2,3,75.0,60.00,100.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
3,4,19.9,53.10,4.0,0,1,1,0,0,TREG-coated,...,2.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
4,5,14.6,54.40,3.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0


In [4]:
# --- 3e. Numeric feature correlations with target ---
numeric_df = df.select_dtypes(include=np.number)
if TARGET_COL in numeric_df.columns:
    corr = numeric_df.corr()[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
    print("Top correlations with target:")
    print(corr)

Top correlations with target:
f_Ms_emu_g_MNP                 0.309831
f_avg_core_width_nm            0.212310
f_area_volume_ratio           -0.157436
f_area_core_estimated_nm2      0.153174
f_H_Oe                         0.139183
f_FCY_kHz                      0.130932
Ca_comp                        0.108651
f_volume_core_estimated_nm3    0.108263
f_conc_MNP_mg_mL              -0.103827
S. No.                        -0.102738
f_is_inorganic                -0.071056
Cu_comp                        0.066102
f_is_organic                   0.053601
Tb_comp                        0.045630
f_Hc_Oe                       -0.041595
Mg_comp                       -0.037980
f_is_composite_coating         0.037516
Ga_comp                       -0.033765
Nd_comp                       -0.031919
Ni_comp                       -0.030614
Mn_comp                        0.029041
f_avg_core_sd_nm _BOTH        -0.025636
Sn_comp                       -0.024635
Co_comp                       -0.020375
f_is_coate

In [5]:
target_col = TARGET_COL.replace('\ufeff', '').replace('\u200b', '').strip()
X = df.drop(columns=[target_col])
y = df[target_col]

# 80 / 10 / 10 split
x_train, x_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print(f"Train : {x_train.shape[0]} rows")
print(f"Val   : {x_val.shape[0]} rows")
print(f"Test  : {x_test.shape[0]} rows  ← locked until final evaluation")

Train : 1238 rows
Val   : 155 rows
Test  : 155 rows  ← locked until final evaluation


In [6]:
# --- 5a. Binary presence indicators (fit-free, safe to apply before split logic) ---
def add_presence_flags(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    df_out['f_is_coated']  = df_out['c_coating'].notna().astype(int)
    df_out['f_is_shaped']  = df_out['c_shape_type'].notna().astype(int)
    df_out['f_is_phased']  = df_out['c_magnetic_phase'].notna().astype(int)
    return df_out

x_train = add_presence_flags(x_train)
x_val   = add_presence_flags(x_val)
x_test  = add_presence_flags(x_test)

In [7]:
# --- 5b. Categorical encoding ---
# Fill NaN → 'Unknown' for unseen categories, then ordinal-encode.
# Where the presence flag == 0 (field was missing), force the encoded value to -1.
# Encoders are fit on train only.

def encode_categorical(train_df, val_df, test_df, cat_col, flag_col):
    for df_part in [train_df, val_df, test_df]:
        df_part[cat_col] = df_part[cat_col].fillna('Unknown').astype(str)

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_df[cat_col] = enc.fit_transform(train_df[[cat_col]])
    val_df[cat_col]   = enc.transform(val_df[[cat_col]])
    test_df[cat_col]  = enc.transform(test_df[[cat_col]])

    for df_part in [train_df, val_df, test_df]:
        df_part.loc[df_part[flag_col] == 0, cat_col] = -1
        df_part.drop(columns=[flag_col], inplace=True)

    return train_df, val_df, test_df

x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_coating',       'f_is_coated')
x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_shape_type',    'f_is_shaped')
x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_magnetic_phase', 'f_is_phased')

print("Encoding done. x_train shape:", x_train.shape)

Encoding done. x_train shape: (1238, 31)


In [8]:
# --- 5c. Numeric imputation (fit on train only) ---
numeric_cols = x_train.select_dtypes(include=np.number).columns.tolist()

imputer = MissForest(rgr=RandomForestRegressor(n_estimators=100,random_state=62),max_iter=10)

if x_train[numeric_cols].isnull().any().any():
    x_train[numeric_cols] = imputer.fit_transform(x_train[numeric_cols])
    x_val[numeric_cols]   = imputer.transform(x_val[numeric_cols])
    x_test[numeric_cols]  = imputer.transform(x_test[numeric_cols])
    print("Imputation applied.")
else:
    print("No missing values in numeric columns — skipping imputation.")

/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
 30%|███       | 3/10 [01:05<02:35, 22.17s/it]/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:303: UserWarning: NRMSE increased.
  warnings.warn("NRMSE increased.")
/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:453: UserWarning: Stopping criterion triggered during fitting. Before last imputation matrix will be returned.
  warnings.warn(
 30%|███       | 3/10 [01:28<03:26, 29.52s/it]
/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/usr/local/lib/python3.13/dist

Imputation applied.


In [9]:
# --- 6a. Feature scaling (QuantileTransformer, fit on train only) ---
transf = QuantileTransformer(random_state=RANDOM_STATE)

x_train_scaled = transf.fit_transform(x_train)
x_val_scaled   = transf.transform(x_val)
x_test_scaled  = transf.transform(x_test)

print("Scaling done. Train shape:", x_train_scaled.shape)

Scaling done. Train shape: (1238, 31)


In [10]:
# --- 6b. Log transformation ---
y_train_log = np.log1p(y_train.values.astype(float))   # log1p on raw SAR
y_val_log   = np.log1p(y_val.values.astype(float))
y_test_log  = np.log1p(y_test.values.astype(float))

scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(y_train_log.reshape(-1, 1)).flatten()
y_val_scaled   = scaler.transform(y_val_log.reshape(-1, 1)).flatten()
y_test_scaled  = scaler.transform(y_test_log.reshape(-1, 1)).flatten()

print("=== Scaler sanity check ===")
print(f"  data_min_ : {scaler.data_min_[0]:.4f}  "
      f"(= log1p({np.expm1(scaler.data_min_[0]):.2f} W/g)")
print(f"  data_max_ : {scaler.data_max_[0]:.4f}  "
      f"(= log1p({np.expm1(scaler.data_max_[0]):.2f} W/g)")
assert scaler.data_max_[0] > 1.0, \
    "Scaler data_max_ is still wrong — y_train may still be pre-transformed. Restart kernel and rerun top-to-bottom."

# Use these as your targets for model training (rename to match rest of notebook)
y_train = y_train_scaled
y_val   = y_val_scaled
y_test  = y_test_scaled


=== Scaler sanity check ===
  data_min_ : 0.1570  (= log1p(0.17 W/g)
  data_max_ : 8.6145  (= log1p(5510.00 W/g)


In [11]:
def inverse_sar(y_scaled: np.ndarray, fitted_scaler: MinMaxScaler) -> np.ndarray:
  y_scaled_clipped = np.clip(y_scaled, 0.0, 1.0)
  pred_log_units      = fitted_scaler.inverse_transform(y_scaled_clipped.reshape(-1, 1)).flatten()
  true_predicted_sar  = np.expm1(pred_log_units)
  return true_predicted_sar

In [12]:
def objective(trial):
    # Define hyperparameters for Random Forest
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 50, 500, step=50),
        max_depth         = trial.suggest_int('max_depth', 3, 20),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 20),
        max_features      = trial.suggest_categorical('max_features', [1.0, 'sqrt', 'log2'])
    )
    
    # Instantiate with the notebook's RANDOM_STATE for reproducibility
    rf = RandomForestRegressor(**params, random_state=RANDOM_STATE)
    
    rf.fit(x_train_scaled, y_train)
    
    preds = rf.predict(x_val_scaled)
    
    rmse = root_mean_squared_error(y_val, inverse_sar(preds, scaler))
    
    return rmse

study = op.create_study(
    direction='minimize',
    sampler=op.samplers.TPESampler(seed=RANDOM_STATE),
    storage='sqlite:///rf_regression_optuna.db',
    load_if_exists=True,
)

# 30-50 trials is usually a good starting point for Random Forests
study.optimize(objective, n_trials=30) 

print("Best params :", study.best_params)
print("Best val RMSE:", round(study.best_value, 4))

[I 2026-10-12 06:28:54,874] A new study created in RDB with name: no-name-4b19d230-d0c5-46d8-8f94-ac5c9ff22ced
[I 2026-10-12 06:28:55,321] Trial 0 finished with value: 131.1923679316264 and parameters: {'n_estimators': 50, 'max_depth': 11, 'min_samples_split': 18, 'min_samples_leaf': 9, 'max_features': 1.0}. Best is trial 0 with value: 131.1923679316264.
[I 2026-10-12 06:28:55,676] Trial 1 finished with value: 96.02324746343378 and parameters: {'n_estimators': 150, 'max_depth': 7, 'min_samples_split': 20, 'min_samples_leaf': 13, 'max_features': 'sqrt'}. Best is trial 1 with value: 96.02324746343378.
[I 2026-10-12 06:28:56,464] Trial 2 finished with value: 90.7311871023796 and parameters: {'n_estimators': 400, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 20, 'max_features': 'log2'}. Best is trial 2 with value: 90.7311871023796.
[I 2026-10-12 06:28:57,351] Trial 3 finished with value: 91.11235275095162 and parameters: {'n_estimators': 450, 'max_depth': 20, 'min_samples_sp

Best params : {'n_estimators': 300, 'max_depth': 3, 'min_samples_split': 11, 'min_samples_leaf': 19, 'max_features': 'sqrt'}
Best val RMSE: 75.4618


In [13]:
# Fit final Random Forest model using the best parameters found
rf = RandomForestRegressor(**study.best_params, random_state=RANDOM_STATE)
rf.fit(x_train_scaled, y_train)

RandomForestRegressor(max_depth=3, max_features='sqrt', min_samples_leaf=19,
                      min_samples_split=11, n_estimators=300, random_state=62)

In [14]:
# Generate predictions using the fitted Random Forest model
y_val_pred  = rf.predict(x_val_scaled)
y_test_pred = rf.predict(x_test_scaled)

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

val_r2    = r2_score(y_val,  y_val_pred)
test_r2   = r2_score(y_test, y_test_pred)
val_rmse  = np.sqrt(mean_squared_error(y_val, y_val_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_q90  = np.quantile(np.abs(y_test - y_test_pred), 0.90)

y_test      = inverse_sar(y_test, scaler)
y_test_pred = inverse_sar(y_test_pred, scaler)
err_90      = np.quantile(np.abs(y_test - y_test_pred), 0.90)

print(f"Validation  — R²: {val_r2:.4f}   RMSE: {val_rmse:.4f}")
print(f"Test        — R²: {test_r2:.4f}   RMSE: {test_rmse:.4f}")
print(f"90th-pct prediction error (test): ±{err_90:.4f}")

Validation  — R²: 0.3520   RMSE: 0.1348
Test        — R²: 0.3265   RMSE: 0.1310
90th-pct prediction error (test): ±219.6455


In [17]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='blue', edgecolor='k', label='Predictions')
min_val = min(np.min(y_test), np.min(y_test_pred))
max_val = max(np.max(y_test), np.max(y_test_pred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal (Actual = Predicted)')
plt.title('Random Forest: Actual vs Predicted SAR', fontsize=14)
plt.xlabel('Actual SAR (W/g)', fontsize=12)
plt.ylabel('Predicted SAR (W/g)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('rf_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.close()

explainer = s.TreeExplainer(rf)
shap_values = explainer.shap_values(x_test_scaled)
plt.figure(figsize=(10, 8))
s.summary_plot(shap_values, x_test_scaled, feature_names=x_test.columns, show=False)
plt.title('Random Forest: SHAP Summary', fontsize=14)
plt.tight_layout()
plt.savefig('rf_shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()